In [1]:
import pandas as pd

df_droid = pd.read_json('../basic/droid_results_real.jsonl', lines=True)
df_raft = pd.read_json('../basic/raft_results_real.jsonl', lines=True)

df_combined = pd.merge(df_droid, df_raft, on='folder')
print(df_combined.head())

                                  folder  droid_score  raft_score
0             base_vlm--flip_pot_upright     0.155326    0.189003
1  base_vlm--move_salt_shaker_onto_plate     0.084985    0.226554
2        dp--cover_white_bowl_with_towel     0.157032    0.319954
3              dp--knock_brown_bear_over     0.107685    0.186541
4               dp--move_corn_onto_plate     0.091503    0.206440


In [2]:
import numpy as np

def entropy(x, bins=30):
    hist, _ = np.histogram(x, bins=bins, range=(0,1), density=True)
    hist = hist + 1e-8
    hist = hist / hist.sum()
    return -np.sum(hist * np.log(hist))

In [3]:
H_d = entropy(df_combined['droid_score'])
H_r = entropy(df_combined['raft_score'])

In [4]:
print(H_d, H_r)

1.7270132811225998 2.3557247614660364


In [5]:
w_d = 1 / H_d
w_r = 1 / H_r

w_sum = w_d + w_r
w_d /= w_sum
w_r /= w_sum

In [6]:
w_d = np.exp(-H_d)
w_r = np.exp(-H_r)

w_sum = w_d + w_r
w_d /= w_sum
w_r /= w_sum

In [7]:
print(w_d, w_r)

0.6521972368950154 0.34780276310498465


In [25]:
summary = df_combined['droid_score'].agg(['std', 'mean']).reset_index()
print(summary)

  index  droid_score
0   std     0.090842
1  mean     0.158861


In [26]:
summary = df_combined['raft_score'].agg(['std', 'mean']).reset_index()
print(summary)

  index  raft_score
0   std    0.528717
1  mean    0.540216


In [14]:
# 평균 0, 표준편차 1로 변환: Z-Score Normalization
df_combined['droid_std'] = (df_combined['droid_score'] - df_combined['droid_score'].mean()) / df_combined['droid_score'].std()
df_combined['raft_std'] = (df_combined['raft_score'] - df_combined['raft_score'].mean()) / df_combined['raft_score'].std()

# 가중치 적용
df_combined['combined_score'] = w_d * df_combined['droid_std'] + w_r * df_combined['raft_std']

In [15]:
print(df_combined.sort_values(by='combined_score', ascending=False).head())

                                     folder  droid_score  raft_score  \
29      openvla--stack_blue_cup_on_pink_cup     0.475736    0.840097   
14                     openvla--lift_cheese     0.219168    2.066266   
16          openvla--move_banana_near_plate     0.288348    1.070630   
5                             dp--pour_corn     0.364966    0.121679   
27  openvla--put_eggplant_into_pot--clutter     0.233252    1.522440   

    combined_score  droid_std  raft_std  
29        2.472253   3.488184  0.567186  
14        1.436840   0.663860  2.886327  
16        1.278558   1.425396  1.003209  
5         1.204390   2.268814 -0.791609  
27        1.180216   0.818901  1.857750  


In [16]:
print(df_combined.sort_values(by='combined_score', ascending=True).head())

                                   folder  droid_score  raft_score  \
12         openvla--knock_brown_bear_over     0.079131    0.097166   
19   openvla--move_salt_shaker_onto_plate     0.080057    0.144662   
9              octo--move_corn_onto_plate     0.094653    0.060503   
8             octo--knock_brown_bear_over     0.094105    0.078640   
1   base_vlm--move_salt_shaker_onto_plate     0.084985    0.226554   

    combined_score  droid_std  raft_std  
12       -0.863864  -0.877672 -0.837972  
19       -0.825971  -0.867477 -0.748139  
9        -0.776547  -0.706811 -0.907315  
8        -0.768550  -0.712843 -0.873011  
1        -0.736725  -0.813236 -0.593251  


In [21]:
df_combined[['folder', 'combined_score']].sort_values(by='combined_score', ascending=False)

,folder,combined_score
29,openvla--stack_blue_cup_on_pink_cup,2.472253
14,openvla--lift_cheese,1.436840
16,openvla--move_banana_near_plate,1.278558
5,dp--pour_corn,1.204390
27,openvla--put_eggplant_into_pot--clutter,1.180216
26,openvla--put_corn_on_plate--clutter,0.594109
23,openvla--put_blue_cup_on_plate--reattempt,0.568332
21,openvla--place_banana_on_plate,0.561882
17,openvla--move_coke_can_near_taylor_swift,0.428908
13,openvla--lift_AAA_battery_fail,0.351576


In [22]:
summary = df_combined['combined_score'].agg(['count', 'mean', 'max']).reset_index()
print(summary)

   index  combined_score
0  count    3.000000e+01
1   mean   -2.072416e-16
2    max    2.472253e+00


In [28]:
import shutil
import os

def copy_videos(df, subdir, video_extension='.mp4'):
    source_dir = '/home/yun/world-model-eval/rollouts/openvla_seeds'
    target_dir = source_dir + "/" + subdir
    
    os.makedirs(target_dir, exist_ok=True)

    copy_count = 0
    for _, row in df.iterrows():
        folder_name = row['folder']
        score = round(row['combined_score'], 4)
        score_str = str(round(row['combined_score'], 4)).replace('.', '_')
        
        matched_files = [
            f for f in os.listdir(source_dir)
            if f == folder_name + video_extension
        ]
        
        if matched_files:
            for f in matched_files:
                src_file_path = os.path.join(source_dir, f)
                dst_file_name = f"{folder_name}_{score_str}{video_extension}"
                dst_file_path = os.path.join(target_dir, dst_file_name)
                shutil.copy2(src_file_path, dst_file_path)
                copy_count += 1
        else:
            print(f"파일 없음: {folder_name}")

    print(f"[{target_dir}] 총 {copy_count}개의 영상 추출 완료!")

In [29]:
copy_videos(df_top10, 'top_anomaly')

[/home/yun/world-model-eval/rollouts/openvla_seeds/top_anomaly] 총 50개의 영상 추출 완료!


In [30]:
df_low10 = df_sorted.groupby('task').tail(10)
print(df_low10)

                             folder  droid_score  raft_score  combined_score  \
13                lift_eggplant_s55     0.148835    0.267060       -0.688107   
22                lift_eggplant_s64     0.138499    0.308301       -0.707028   
27                lift_eggplant_s69     0.137634    0.301621       -0.718225   
25                lift_eggplant_s67     0.141980    0.266378       -0.727264   
4                 lift_eggplant_s46     0.131486    0.306183       -0.748435   
19                lift_eggplant_s61     0.125882    0.334788       -0.752764   
26                lift_eggplant_s68     0.131825    0.284561       -0.767050   
16                lift_eggplant_s58     0.130546    0.246555       -0.810308   
1                 lift_eggplant_s43     0.110947    0.280372       -0.888314   
0                 lift_eggplant_s42     0.095967    0.216199       -1.033376   
35        put_blue_cup_on_plate_s47     0.191029    0.644425       -0.092896   
34        put_blue_cup_on_plate_s46     

In [31]:
copy_videos(df_low10, 'low_anomaly')

[/home/yun/world-model-eval/rollouts/openvla_seeds/low_anomaly] 총 50개의 영상 추출 완료!
